## WORKSHOP 2- Object detection using web camera
## NAME: HARINI S
## REG NO-212224240049

In [4]:
import cv2
import numpy as np
import time

# Model files
PROTOTXT = "deploy.prototxt"
MODEL = "mobilenet_iter_73000.caffemodel"

# Class labels
CLASSES = ["background","aeroplane","bicycle","bird","boat",
 "bottle","bus","car","cat","chair","cow","diningtable",
 "dog","horse","motorbike","person","pottedplant",
 "sheep","sofa","train","tvmonitor"]

CONF_THRESH = 0.5
FONT = cv2.FONT_HERSHEY_SIMPLEX

# Load model
net = cv2.dnn.readNetFromCaffe(PROTOTXT, MODEL)

# Open webcam
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    raise SystemExit("Webcam not accessible")

fps_time = time.time()
frame_count = 0

try:
    while True:
        ret, frame = cap.read()

        if not ret:
            break

        (h, w) = frame.shape[:2]

        # Create blob
        blob = cv2.dnn.blobFromImage(
            cv2.resize(frame, (300,300)),
            0.007843,
            (300,300),
            127.5
        )

        net.setInput(blob)
        detections = net.forward()

        # Draw detections
        for i in range(detections.shape[2]):

            conf = float(detections[0,0,i,2])

            if conf > CONF_THRESH:

                idx = int(detections[0,0,i,1])

                box = detections[0,0,i,3:7] * np.array([w,h,w,h])

                (startX, startY, endX, endY) = box.astype("int")

                label = f"{CLASSES[idx]}: {conf:.2f}"

                cv2.rectangle(frame,
                              (startX,startY),
                              (endX,endY),
                              (0,255,0), 2)

                y = startY - 15 if startY > 30 else startY + 15

                cv2.putText(frame,
                            label,
                            (startX,y),
                            FONT,
                            0.5,
                            (0,255,0),
                            2)

        # Show output
        cv2.imshow("Object Detection", frame)

        # Press q to quit
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

finally:
    cap.release()
    cv2.destroyAllWindows()